# Library Implementation

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

#(MNIST-like digits)
digits = datasets.load_digits()
X = digits.data
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Step (b): Select 100 labeled samples
n_labeled = 100
indices = np.random.permutation(len(X_train))

labeled_idx = indices[:n_labeled]
unlabeled_idx = indices[n_labeled:]

# Step (c): Create semi-supervised labels
y_semi = np.copy(y_train)
y_semi[unlabeled_idx] = -1   # unlabeled = -1

X_labeled = X_train[labeled_idx]
y_labeled = y_train[labeled_idx]

X_unlabeled = X_train[unlabeled_idx]
y_unlabeled_true = y_train[unlabeled_idx]  # ground truth for evaluation

In [ ]:
# FIX: Normalize data before using SVC-based SelfTrainingClassifier
# Without scaling, SVC on raw pixel data performs near-randomly (~10%)
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_unlabeled_scaled = scaler.transform(X_unlabeled)

# Use gamma='scale' (better default than 'auto' for normalized data)
base_model = SVC(probability=True, kernel='rbf', gamma='scale')

self_model = SelfTrainingClassifier(base_model)
self_model.fit(X_train_scaled, y_semi)

# Predict pseudo-labels for unlabeled data
pseudo_labels = self_model.predict(X_unlabeled_scaled)

acc = accuracy_score(y_unlabeled_true, pseudo_labels)
print("Self-Training Accuracy:", acc)

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB
from mvlearn.semi_supervised import CTClassifier

digits = datasets.load_digits()
X = digits.data
y = digits.target

# Binary filter: CTClassifier only supports binary classification
# Note: mvlearn's CTClassifier is limited to 2 classes
mask = (y == 0) | (y == 1)
X = X[mask]
y = y[mask]

# Split into 2 views (each half of features becomes one view)
X1 = X[:, :32]
X2 = X[:, 32:]

# Train-test split
X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    X1, X2, y, test_size=0.3, random_state=42
)

# Create unlabeled indices
np.random.seed(42)
n_unlabeled = int(0.5 * len(y_train))
unlabeled_idx = np.random.choice(len(y_train), n_unlabeled, replace=False)

# CTClassifier expects NaN for unlabeled samples
y_semi = y_train.astype(float)
y_semi[unlabeled_idx] = np.nan

# Models for each view
clf1 = GaussianNB()
clf2 = GaussianNB()

# Co-Training
ct = CTClassifier(clf1, clf2)
ct.fit([X1_train, X2_train], y_semi)

# Predict pseudo-labels for unlabeled data
pseudo_labels = ct.predict([
    X1_train[unlabeled_idx],
    X2_train[unlabeled_idx]
])

# Accuracy
acc = accuracy_score(y_train[unlabeled_idx], pseudo_labels)
print("Co-Training Accuracy:", acc)

In [ ]:
from sklearn.cluster import KMeans

# Re-use X_train, y_train, X_unlabeled, y_unlabeled_true from Cell 1
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_train)

cluster_labels = {}

for i in range(10):
    points = np.where(clusters == i)[0]
    labels = y_train[points]

    if len(labels) > 0:
        # Majority label within the cluster
        cluster_labels[i] = np.bincount(labels).argmax()

# Predict pseudo-labels for unlabeled data
unlabeled_clusters = kmeans.predict(X_unlabeled)
pseudo_labels = np.array([cluster_labels[c] for c in unlabeled_clusters])

acc = accuracy_score(y_unlabeled_true, pseudo_labels)
print("Clustering Accuracy:", acc)

# Manual Implementation

In [ ]:
def knn(X_train, y_train, X_test, k=3):
    preds = []
    for x in X_test:
        dist = np.linalg.norm(X_train - x, axis=1)
        idx = dist.argsort()[:k]
        labels = y_train[idx]
        preds.append(np.bincount(labels).argmax())
    return np.array(preds)

# Train on labeled data, predict pseudo-labels for unlabeled data
pseudo_labels = knn(X_labeled, y_labeled, X_unlabeled)

acc = accuracy_score(y_unlabeled_true, pseudo_labels)
print("Manual Self-Training Accuracy:", acc)

In [ ]:
# Split features into two views (each view = half the features)
X1 = X_train[:, :32]
X2 = X_train[:, 32:]

X1_l = X1[labeled_idx]
X2_l = X2[labeled_idx]
y_l = y_labeled

X1_u = X1[unlabeled_idx]
X2_u = X2[unlabeled_idx]

# Each view's model independently predicts
pred1 = knn(X1_l, y_l, X1_u)
pred2 = knn(X2_l, y_l, X2_u)

# FIX: Combine predictions using confidence-based agreement
# When both views agree, use that label.
# When they disagree, use the view with more nearest-neighbor agreement
# (here approximated by majority vote across both predictions).
final_pred = []
for i in range(len(pred1)):
    if pred1[i] == pred2[i]:
        final_pred.append(pred1[i])      # both views agree
    else:
        # Disagreement: pick via majority across all k-NN neighbours combined
        combined = np.concatenate([
            knn(X1_l, y_l, X1_u[i:i+1]),
            knn(X2_l, y_l, X2_u[i:i+1])
        ])
        final_pred.append(np.bincount(combined).argmax())

final_pred = np.array(final_pred)

acc = accuracy_score(y_unlabeled_true, final_pred)
print("Manual Co-Training Accuracy:", acc)

In [ ]:
def kmeans_manual(X, k=10, iters=10):
    np.random.seed(42)
    centroids = X[np.random.choice(len(X), k, replace=False)]

    for _ in range(iters):
        dist = np.linalg.norm(X[:, None] - centroids, axis=2)
        clusters = np.argmin(dist, axis=1)

        new_centroids = []
        for i in range(k):
            pts = X[clusters == i]
            if len(pts) > 0:
                new_centroids.append(pts.mean(axis=0))
            else:
                new_centroids.append(centroids[i])
        centroids = np.array(new_centroids)

    return clusters, centroids


clusters, centroids = kmeans_manual(X_train)

cluster_labels = {}
for i in range(10):
    labels = y_train[clusters == i]
    if len(labels) > 0:
        cluster_labels[i] = np.bincount(labels).argmax()

# Assign pseudo-labels to unlabeled data based on nearest centroid
dist = np.linalg.norm(X_unlabeled[:, None] - centroids, axis=2)
unlabeled_clusters = np.argmin(dist, axis=1)

pseudo_labels = np.array([cluster_labels[c] for c in unlabeled_clusters])

acc = accuracy_score(y_unlabeled_true, pseudo_labels)
print("Manual KMeans Accuracy:", acc)